In [ ]:
# Re-import needed libraries after reset
import os
import shutil
from sklearn.model_selection import train_test_split
import pandas as pd

# ✅ Paths to the new dataset folders (7 class folders)
parent_dir = "/content/drive/MyDrive/hand-picked-data"
class_folders = [os.path.join(parent_dir, folder) for folder in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, folder))]

# ✅ Temporary merged directory
merged_dir = "/content/drive/MyDrive/hand-picked-data/dataset"
merged_images = os.path.join(merged_dir, "images")
merged_labels = os.path.join(merged_dir, "labels")
os.makedirs(merged_images, exist_ok=True)
os.makedirs(merged_labels, exist_ok=True)

# ✅ Merge images and labels
for folder in class_folders:
    img_dir = os.path.join(folder, "images")
    lbl_dir = os.path.join(folder, "labels")
    for fname in os.listdir(img_dir):
        if fname.endswith(".jpg"):
            shutil.copy(os.path.join(img_dir, fname), os.path.join(merged_images, fname))
            label_name = fname.replace(".jpg", ".txt")
            lbl_path = os.path.join(lbl_dir, label_name)
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, os.path.join(merged_labels, label_name))

# ✅ Split into 80:20
all_images = [f for f in os.listdir(merged_images) if f.endswith(".jpg")]
train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

# ✅ Output directories
split_base = "/content/drive/MyDrive/hand-picked-data/Final_Split_New_Datasets"
train_img_dir = os.path.join(split_base, "train/images")
train_lbl_dir = os.path.join(split_base, "train/labels")
val_img_dir = os.path.join(split_base, "val/images")
val_lbl_dir = os.path.join(split_base, "val/labels")

os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

# ✅ Copy files
for f in train_files:
    shutil.copy(os.path.join(merged_images, f), os.path.join(train_img_dir, f))
    shutil.copy(os.path.join(merged_labels, f.replace(".jpg", ".txt")), os.path.join(train_lbl_dir, f.replace(".jpg", ".txt")))

for f in val_files:
    shutil.copy(os.path.join(merged_images, f), os.path.join(val_img_dir, f))
    shutil.copy(os.path.join(merged_labels, f.replace(".jpg", ".txt")), os.path.join(val_lbl_dir, f.replace(".jpg", ".txt")))

# ✅ Summary
summary = pd.DataFrame({
    "Total Images": [len(all_images)],
    "Training Set": [len(train_files)],
    "Validation Set": [len(val_files)],
    "Class Folders Merged": [len(class_folders)]
})

import pandas as pd
from IPython.display import display

summary = pd.DataFrame({
    "Total Clean Images": [len(all_images)],
    "Training Set": [len(train_imgs)],
    "Validation Set": [len(val_imgs)],
    "Faulty Images Removed": [len(faulty_ids)]
})

display(summary)


KeyboardInterrupt: 

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
import pandas as pd
from IPython.display import display

# ✅ Root path where class folders are located
parent_dir = "/content/drive/MyDrive/hand-picked-data"
merged_dir = os.path.join(parent_dir, "dataset")

# ✅ Clean up old merged folder if it exists
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir)

# ✅ Recreate merged dataset folders
merged_images = os.path.join(merged_dir, "images")
merged_labels = os.path.join(merged_dir, "labels")
os.makedirs(merged_images, exist_ok=True)
os.makedirs(merged_labels, exist_ok=True)

# ✅ Prepare list of valid class folders (exclude "dataset")
class_folders = [
    os.path.join(parent_dir, folder)
    for folder in os.listdir(parent_dir)
    if os.path.isdir(os.path.join(parent_dir, folder)) and folder != "dataset"
]

print("📦 Merging image-label pairs...")

merged_count = 0
skipped_missing_label = 0
skipped_structure = 0

for folder in class_folders:
    img_dir = os.path.join(folder, "images")
    lbl_dir = os.path.join(folder, "labels")

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f"⚠️ Skipped: {folder} — Missing 'images/' or 'labels/'")
        skipped_structure += 1
        continue

    for fname in os.listdir(img_dir):
        if fname.endswith(".jpg"):
            img_src = os.path.join(img_dir, fname)
            lbl_src = os.path.join(lbl_dir, fname.replace(".jpg", ".txt"))

            if not os.path.exists(lbl_src):
                skipped_missing_label += 1
                continue

            shutil.copy(img_src, os.path.join(merged_images, fname))
            shutil.copy(lbl_src, os.path.join(merged_labels, fname.replace(".jpg", ".txt")))
            merged_count += 1

print(f"✅ Merged {merged_count} image-label pairs")
print(f"⚠️ Skipped {skipped_missing_label} files with missing labels")
print(f"⚠️ Skipped {skipped_structure} folders with missing structure")

# ✅ Split merged dataset into train/val (80:20)
all_images = [f for f in os.listdir(merged_images) if f.endswith(".jpg")]
train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

# ✅ Create final output split directories
split_base = os.path.join(parent_dir, "Final_Split_New_Datasets")
for split in ["train/images", "train/labels", "val/images", "val/labels"]:
    os.makedirs(os.path.join(split_base, split), exist_ok=True)

# ✅ Copy to train
for f in train_files:
    img_path = os.path.join(merged_images, f)
    lbl_path = os.path.join(merged_labels, f.replace(".jpg", ".txt"))
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        shutil.copy(img_path, os.path.join(split_base, "train/images", f))
        shutil.copy(lbl_path, os.path.join(split_base, "train/labels", f.replace(".jpg", ".txt")))

# ✅ Copy to val
for f in val_files:
    img_path = os.path.join(merged_images, f)
    lbl_path = os.path.join(merged_labels, f.replace(".jpg", ".txt"))
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        shutil.copy(img_path, os.path.join(split_base, "val/images", f))
        shutil.copy(lbl_path, os.path.join(split_base, "val/labels", f.replace(".jpg", ".txt")))

# ✅ Summary
summary = pd.DataFrame({
    "Total Merged Images": [len(all_images)],
    "Training Set": [len(train_files)],
    "Validation Set": [len(val_files)],
    "Valid Class Folders": [len(class_folders) - skipped_structure],
    "Skipped (missing labels)": [skipped_missing_label]
})

display(summary)


📦 Merging image-label pairs...
⚠️ Skipped: /content/drive/MyDrive/hand-picked-data/images — Missing 'images/' or 'labels/'
⚠️ Skipped: /content/drive/MyDrive/hand-picked-data/labels — Missing 'images/' or 'labels/'
⚠️ Skipped: /content/drive/MyDrive/hand-picked-data/Final_Split_New_Datasets — Missing 'images/' or 'labels/'
✅ Merged 1388 image-label pairs
⚠️ Skipped 0 files with missing labels
⚠️ Skipped 3 folders with missing structure


,Total Merged Images,Training Set,Validation Set,Valid Class Folders,Skipped (missing labels)
0,1388,1110,278,7,0


In [ ]:
import os

# Define training script content
from ultralytics import YOLO
import os
import cv2
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# ✅ Paths
yaml_path = "/content/drive/MyDrive/hand-picked-data/Final_Split_New_Datasets/config.yaml"
model_path = "models/yolo11n.pt"
output_dir = "/content/drive/MyDrive/YOLOv11_Results/Handpicked_YOLOv11_1000x1000"

val_images_path = "/content/drive/MyDrive/hand-picked-data/Final_Split_New_Datasets/val/images"
val_labels_path = "/content/drive/MyDrive/hand-picked-data/Final_Split_New_Datasets/val/labels"

# ✅ Train YOLOv11
model = YOLO(model_path)
model.train(
    data=yaml_path,
    epochs=300,
    imgsz=1000,
    patience=30,
    device="cuda",
    project=output_dir,
    name="train_1000x1000",
    save=True,
    resume=True
)

# ✅ Evaluation
val_results = model.val(data=yaml_path, save_json=True)

# ✅ Output folders for errors
fp_dir = os.path.join(output_dir, "false_positives")
fn_dir = os.path.join(output_dir, "false_negatives")
mc_dir = os.path.join(output_dir, "misclassified")
os.makedirs(fp_dir, exist_ok=True)
os.makedirs(fn_dir, exist_ok=True)
os.makedirs(mc_dir, exist_ok=True)

# ✅ Class names
class_names = {
    0: "Formicidae", 1: "Brachycera", 2: "Nematocera", 3: "Collembola",
    4: "Arachnida", 5: "Aphididae", 6: "Hymenoptera", 7: "Syrphidae"
}

# ✅ IOU calculation
def compute_iou(box1, box2):
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

print("🔍 Evaluating validation set...")
y_true = []
y_pred = []

for img_name in tqdm(os.listdir(val_images_path)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace(".jpg", ".txt"))
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    results = model(img_path)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()
    pred_classes = results.boxes.cls.cpu().numpy()

    gt_boxes = []
    gt_classes = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w / 2) * width)
                y1 = int((y - h / 2) * height)
                x2 = int((x + w / 2) * width)
                y2 = int((y + h / 2) * height)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    matched_pred = set()
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        px1, py1, px2, py2 = map(int, pbox[:4])
        pred_cls = int(pred_classes[i])
        for j, gtbox in enumerate(gt_boxes):
            iou = compute_iou(pbox[:4], gtbox)
            if iou > 0.3:
                matched_pred.add(i)
                matched_gt.add(j)
                if pred_cls != gt_classes[j]:
                    img_copy = image.copy()
                    gx1, gy1, gx2, gy2 = gtbox
                    cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                    cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                    cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls, pred_cls)}", (px1, py1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j], gt_classes[j])}", (gx1, gy1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    cv2.imwrite(os.path.join(mc_dir, f"mc_{img_name}"), img_copy)
                break

    for i, pbox in enumerate(pred_boxes):
        if i not in matched_pred:
            y_true.append(0)
            y_pred.append(1)
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            img_copy = image.copy()
            cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
            cv2.putText(img_copy, f"FP: {class_names.get(pred_cls, pred_cls)}", (px1, py1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.imwrite(os.path.join(fp_dir, f"fp_{img_name}"), img_copy)

    for j, gtbox in enumerate(gt_boxes):
        if j not in matched_gt:
            y_true.append(1)
            y_pred.append(0)
            gx1, gy1, gx2, gy2 = gtbox
            img_copy = image.copy()
            cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
            cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            cv2.imwrite(os.path.join(fn_dir, f"fn_{img_name}"), img_copy)

# ✅ Metrics
precision = precision_score(y_true, y_pred) * 100
recall = recall_score(y_true, y_pred) * 100
f1 = f1_score(y_true, y_pred) * 100

print(f"✅ Evaluation Complete:")
print(f"Precision: {precision:.2f}%")
print(f"Recall:    {recall:.2f}%")
print(f"F1 Score:  {f1:.2f}%")
print(f"→ False Positives saved to: {fp_dir}")
print(f"→ False Negatives saved to: {fn_dir}")
print(f"→ Misclassifications saved to: {mc_dir}")


# Save to .py script so user can run it
script_path = "/mnt/data/train_and_eval_yolov11n_1000x1000.py"
with open(script_path, "w") as f:
    f.write(script_content)

script_path


In [ ]:
import os

# Set base directory for dataset
base_dir = "/content/drive/MyDrive/hand-picked-data"

# Extension filters
image_ext = ".jpg"
label_ext = ".txt"

report = []

# Loop through each class folder
for class_name in os.listdir(base_dir):
    class_path = os.path.join(base_dir, class_name)
    if not os.path.isdir(class_path):
        continue

    img_dir = os.path.join(class_path, "images")
    lbl_dir = os.path.join(class_path, "labels")

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f"⚠️ Missing 'images/' or 'labels/' in: {class_name}")
        continue

    img_files = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith(image_ext))
    lbl_files = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith(label_ext))

    unmatched_imgs = img_files - lbl_files
    unmatched_lbls = lbl_files - img_files
    matched = img_files & lbl_files

    report.append({
        "Class": class_name,
        "Image Count": len(img_files),
        "Label Count": len(lbl_files),
        "Matched": len(matched),
        "Images w/o Labels": len(unmatched_imgs),
        "Labels w/o Images": len(unmatched_lbls)
    })

    if unmatched_imgs:
        print(f"🖼️ Images without labels in '{class_name}': {sorted(unmatched_imgs)}")
    if unmatched_lbls:
        print(f"📝 Labels without images in '{class_name}': {sorted(unmatched_lbls)}")

import pandas as pd
df = pd.DataFrame(report)
print("\n📊 Dataset Integrity Summary:")
print(df)


📝 Labels without images in 'Arachnida': ['11_364515', '11_364516', '11_364517', '11_364518', '11_364519', '11_364521', '11_364522', '11_364523', '11_364524', '11_364525', '11_364528', '11_364530', '11_364531', '11_364532', '11_364533', '11_364534', '11_364535', '11_364536', '11_364537', '11_364538', '11_364539', '11_364540', '11_364541', '11_364542', '11_364543', '11_364544', '11_364545', '11_364546', '11_364547', '11_364548', '11_364549', '11_364550', '11_364553', '11_364554', '11_364555', '11_364560', '11_364586', '11_364588', '11_364596', '11_364607', '11_364619', '11_364620', '11_364628', '11_364631', '11_364632', '11_364633', '11_364639', '11_364655', '11_364658', '11_364666', '11_364669', '265_489208', '265_769887', '793_1801365', '793_1801375', '827_1542128', '827_1542129', '827_1542130', '827_1542133', '827_1542137', '827_1542139', '827_1542140', '827_1542141', '827_1542144', '827_1542145', '827_1542147', '827_1542149', '827_1591071']
📝 Labels without images in 'Brachycera': ['

In [ ]:
import os

# ✅ Path to your dataset containing the class folders
base_dir = "/content/drive/MyDrive/hand-picked-data"

# ✅ Classes you want to process (excluding "dataset" and any already merged ones)
class_folders = [d for d in os.listdir(base_dir)
                 if os.path.isdir(os.path.join(base_dir, d)) and d != "dataset"]

removed = {}

for class_name in class_folders:
    lbl_dir = os.path.join(base_dir, class_name, "labels")
    img_dir = os.path.join(base_dir, class_name, "images")

    if not os.path.exists(lbl_dir) or not os.path.exists(img_dir):
        continue

    label_files = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]
    img_files = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.endswith(".jpg"))

    count = 0
    for label_file in label_files:
        label_id = os.path.splitext(label_file)[0]
        if label_id not in img_files:
            os.remove(os.path.join(lbl_dir, label_file))
            count += 1

    removed[class_name] = count

# ✅ Report
print("\n🧹 Removed orphan labels:")
for k, v in removed.items():
    print(f"{k}: {v} orphan label files removed")



🧹 Removed orphan labels:
Arachnida: 68 orphan label files removed
Brachycera: 23 orphan label files removed
Apoidea: 20 orphan label files removed
Formicidae: 10 orphan label files removed
Coleoptera: 25 orphan label files removed
Syraphidae: 2 orphan label files removed
Nematocera: 121 orphan label files removed


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

# ✅ Update this to your actual dataset base path
base_path = "/content/drive/MyDrive/hand-picked-data"

# ✅ Class folder to ID mapping
class_mapping = {
    "Apoidea": 0,
    "Arachnida": 1,
    "Brachycera": 2,
    "Coleoptera": 3,
    "Formicidae": 4,
    "Formicidae": 5,
    "Syraphidae": 6
}

# ✅ Output merged & split paths
output_base = os.path.join(base_path, "2024-04-11-reindexed_and_split")
merged_images_dir = os.path.join(output_base, "merged/images")
merged_labels_dir = os.path.join(output_base, "merged/labels")
split_train_img = os.path.join(output_base, "train/images")
split_train_lbl = os.path.join(output_base, "train/labels")
split_val_img = os.path.join(output_base, "val/images")
split_val_lbl = os.path.join(output_base, "val/labels")

# ✅ Create directories
for d in [merged_images_dir, merged_labels_dir, split_train_img, split_train_lbl, split_val_img, split_val_lbl]:
    os.makedirs(d, exist_ok=True)

# ✅ Reindex and copy
print("🔁 Rewriting label files with new indices...")
for class_name, class_id in class_mapping.items():
    class_path = os.path.join(base_path, class_name)
    img_dir = os.path.join(class_path, "images")
    lbl_dir = os.path.join(class_path, "labels")

    for fname in tqdm(os.listdir(img_dir), desc=f"Processing {class_name}"):
        if not fname.endswith(".jpg"):
            continue
        img_src = os.path.join(img_dir, fname)
        lbl_src = os.path.join(lbl_dir, fname.replace(".jpg", ".txt"))

        img_dst = os.path.join(merged_images_dir, fname)
        lbl_dst = os.path.join(merged_labels_dir, fname.replace(".jpg", ".txt"))

        shutil.copy(img_src, img_dst)

        if os.path.exists(lbl_src):
            with open(lbl_src, 'r') as f:
                lines = f.readlines()

            with open(lbl_dst, 'w') as f_out:
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        parts[0] = str(class_id)
                        f_out.write(" ".join(parts) + "\n")

print("✅ Reindexing and merging completed.")

# ✅ 80:20 Train-Val Split
all_images = [f for f in os.listdir(merged_images_dir) if f.endswith(".jpg")]
train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)

# ✅ Copy train files
for f in train_imgs:
    shutil.copy(os.path.join(merged_images_dir, f), os.path.join(split_train_img, f))
    shutil.copy(os.path.join(merged_labels_dir, f.replace(".jpg", ".txt")), os.path.join(split_train_lbl, f.replace(".jpg", ".txt")))

# ✅ Copy val files
for f in val_imgs:
    shutil.copy(os.path.join(merged_images_dir, f), os.path.join(split_val_img, f))
    shutil.copy(os.path.join(merged_labels_dir, f.replace(".jpg", ".txt")), os.path.join(split_val_lbl, f.replace(".jpg", ".txt")))

# ✅ Summary
summary = pd.DataFrame({
    "Total Merged Images": [len(all_images)],
    "Training Set": [len(train_imgs)],
    "Validation Set": [len(val_imgs)],
    "Classes Processed": [len(class_mapping)]
})
print("\n📊 Dataset Preparation Summary:")
print(summary.to_string(index=False))


🔁 Rewriting label files with new indices...


Processing Syraphidae: 100%|██████████| 133/133 [00:34<00:00,  3.83it/s]


✅ Reindexing and merging completed.

📊 Dataset Preparation Summary:
 Total Merged Images  Training Set  Validation Set  Classes Processed
                1388          1110             278                  7


In [ ]:
!pip install ultralytics
import os
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.metrics import precision_score, recall_score, f1_score

# ✅ Paths
yaml_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml"
model_path = "models/yolo11n.pt"
output_dir = "/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run"

# ✅ Train YOLOv11n model (no resume)
model = YOLO(model_path)
model.train(
    data=yaml_path,
    epochs=300,
    imgsz=1000,
    patience=20,
    save=True,
    project=output_dir,
    name="train_yolo11n_reindexed"
)

# ✅ Validation and Evaluation
val_images_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images"
val_labels_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/labels"

fp_dir = os.path.join(output_dir, "false_positives")
fn_dir = os.path.join(output_dir, "false_negatives")
mc_dir = os.path.join(output_dir, "misclassified")
os.makedirs(fp_dir, exist_ok=True)
os.makedirs(fn_dir, exist_ok=True)
os.makedirs(mc_dir, exist_ok=True)

class_names = {
    0: "Apoidea", 1: "Arachnida", 2: "Brachycera", 3: "Coleoptera",
    4: "Formicidae", 5: "Nematocera", 6: "Syraphidae"
}

def compute_iou(box1, box2):
    xA, yA = max(box1[0], box2[0]), max(box1[1], box2[1])
    xB, yB = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

print("🔍 Evaluating on validation set...")
y_true, y_pred = [], []

for img_name in tqdm(os.listdir(val_images_path)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace(".jpg", ".txt"))
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    results = model(img_path, conf=0.25, iou=0.7)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()
    pred_classes = results.boxes.cls.cpu().numpy()

    gt_boxes, gt_classes = [], []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w/2) * width)
                y1 = int((y - h/2) * height)
                x2 = int((x + w/2) * width)
                y2 = int((y + h/2) * height)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    matched_pred = set()
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        px1, py1, px2, py2 = map(int, pbox[:4])
        pred_cls = int(pred_classes[i])
        for j, gtbox in enumerate(gt_boxes):
            iou = compute_iou(pbox[:4], gtbox)
            if iou > 0.3:
                matched_pred.add(i)
                matched_gt.add(j)
                if pred_cls != gt_classes[j]:
                    img_copy = image.copy()
                    gx1, gy1, gx2, gy2 = gtbox
                    cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                    cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                    cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                    cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j])}", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                    cv2.imwrite(os.path.join(mc_dir, f"mc_{img_name}"), img_copy)
                break

    for i, pbox in enumerate(pred_boxes):
        if i not in matched_pred:
            y_true.append(0)
            y_pred.append(1)
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            img_copy = image.copy()
            cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
            cv2.putText(img_copy, f"FP: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv2.imwrite(os.path.join(fp_dir, f"fp_{img_name}"), img_copy)

    for j, gtbox in enumerate(gt_boxes):
        if j not in matched_gt:
            y_true.append(1)
            y_pred.append(0)
            gx1, gy1, gx2, gy2 = gtbox
            img_copy = image.copy()
            cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
            cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            cv2.imwrite(os.path.join(fn_dir, f"fn_{img_name}"), img_copy)

# ✅ Summary metrics
precision = precision_score(y_true, y_pred) * 100
recall = recall_score(y_true, y_pred) * 100
f1 = f1_score(y_true, y_pred) * 100

print("\n✅ Evaluation Summary:")
print(f"Precision: {precision:.2f}%")
print(f"Recall:    {recall:.2f}%")
print(f"F1 Score:  {f1:.2f}%")
print(f"→ FP saved to: {fp_dir}")
print(f"→ FN saved to: {fn_dir}")
print(f"→ MC saved to: {mc_dir}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.1/994.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

100%|██████████| 5.35M/5.35M [00:00<00:00, 124MB/s]


Ultralytics 8.3.104 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=models/yolo11n.pt, data=/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml, epochs=300, time=None, patience=20, batch=16, imgsz=1000, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run, name=train_yolo11n_reindexed, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False

100%|██████████| 755k/755k [00:00<00:00, 19.6MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

...


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/1_310899.jpg: 800x1024 1 Coleoptera, 1 Syraphidae, 12.8ms
Speed: 6.4ms preprocess, 12.8ms inference, 1.3ms postprocess per image at shape (1, 3, 800, 1024)

image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/265_935599.jpg: 1024x1024 1 Apoidea, 1 Syraphidae, 15.1ms
Speed: 7.2ms preprocess, 15.1ms inference, 1.5ms postprocess per image at shape (1, 3, 1024, 1024)


100%|█████████▉| 277/278 [00:44<00:00, 11.42it/s]


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/793_1870086.jpg: 1024x1024 1 Coleoptera, 14.1ms
Speed: 7.2ms preprocess, 14.1ms inference, 1.4ms postprocess per image at shape (1, 3, 1024, 1024)


100%|██████████| 278/278 [00:44<00:00,  6.20it/s]


✅ Evaluation Summary:
Precision: 0.00%
Recall:    0.00%
F1 Score:  0.00%
→ FP saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run/false_positives
→ FN saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run/false_negatives
→ MC saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run/misclassified


In [ ]:
!pip install ultralytics
import os
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.metrics import precision_score, recall_score, f1_score

# ✅ Paths
yaml_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml"
model_path = "models/yolo11n.pt"
output_dir = "/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_1200"

# ✅ Train YOLOv11n model (no resume)
model = YOLO(model_path)
model.train(
    data=yaml_path,
    epochs=300,
    imgsz=1200,
    patience=20,
    save=True,
    project=output_dir,
    name="train_yolo11n_reindexed_1200"
)

# ✅ Validation and Evaluation
val_images_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images"
val_labels_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/labels"

fp_dir = os.path.join(output_dir, "false_positives")
fn_dir = os.path.join(output_dir, "false_negatives")
mc_dir = os.path.join(output_dir, "misclassified")
os.makedirs(fp_dir, exist_ok=True)
os.makedirs(fn_dir, exist_ok=True)
os.makedirs(mc_dir, exist_ok=True)

class_names = {
    0: "Apoidea", 1: "Arachnida", 2: "Brachycera", 3: "Coleoptera",
    4: "Formicidae", 5: "Nematocera", 6: "Syraphidae"
}

def compute_iou(box1, box2):
    xA, yA = max(box1[0], box2[0]), max(box1[1], box2[1])
    xB, yB = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

print("🔍 Evaluating on validation set...")
y_true, y_pred = [], []

for img_name in tqdm(os.listdir(val_images_path)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace(".jpg", ".txt"))
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    results = model(img_path, conf=0.25, iou=0.7)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()
    pred_classes = results.boxes.cls.cpu().numpy()

    gt_boxes, gt_classes = [], []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w/2) * width)
                y1 = int((y - h/2) * height)
                x2 = int((x + w/2) * width)
                y2 = int((y + h/2) * height)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    matched_pred = set()
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        px1, py1, px2, py2 = map(int, pbox[:4])
        pred_cls = int(pred_classes[i])
        for j, gtbox in enumerate(gt_boxes):
            iou = compute_iou(pbox[:4], gtbox)
            if iou > 0.3:
                matched_pred.add(i)
                matched_gt.add(j)
                if pred_cls != gt_classes[j]:
                    img_copy = image.copy()
                    gx1, gy1, gx2, gy2 = gtbox
                    cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                    cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                    cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                    cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j])}", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                    cv2.imwrite(os.path.join(mc_dir, f"mc_{img_name}"), img_copy)
                break

    for i, pbox in enumerate(pred_boxes):
        if i not in matched_pred:
            y_true.append(0)
            y_pred.append(1)
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            img_copy = image.copy()
            cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
            cv2.putText(img_copy, f"FP: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv2.imwrite(os.path.join(fp_dir, f"fp_{img_name}"), img_copy)

    for j, gtbox in enumerate(gt_boxes):
        if j not in matched_gt:
            y_true.append(1)
            y_pred.append(0)
            gx1, gy1, gx2, gy2 = gtbox
            img_copy = image.copy()
            cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
            cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            cv2.imwrite(os.path.join(fn_dir, f"fn_{img_name}"), img_copy)

# ✅ Summary metrics
precision = precision_score(y_true, y_pred) * 100
recall = recall_score(y_true, y_pred) * 100
f1 = f1_score(y_true, y_pred) * 100

print("\n✅ Evaluation Summary:")
print(f"Precision: {precision:.2f}%")
print(f"Recall:    {recall:.2f}%")
print(f"F1 Score:  {f1:.2f}%")
print(f"→ FP saved to: {fp_dir}")
print(f"→ FN saved to: {fn_dir}")
print(f"→ MC saved to: {mc_dir}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.1/994.1 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

100%|██████████| 5.35M/5.35M [00:00<00:00, 261MB/s]


Ultralytics 8.3.104 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: task=detect, mode=train, model=models/yolo11n.pt, data=/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml, epochs=300, time=None, patience=20, batch=16, imgsz=1200, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_1200, name=train_yolo11n_reindexed_1200, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_

100%|██████████| 755k/755k [00:00<00:00, 126MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

...


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/265_935599.jpg: 1216x1216 1 Brachycera, 1 Syraphidae, 10.8ms
Speed: 9.9ms preprocess, 10.8ms inference, 1.4ms postprocess per image at shape (1, 3, 1216, 1216)


100%|█████████▉| 277/278 [04:15<00:00,  1.17it/s]


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/793_1870086.jpg: 1216x1216 2 Coleopteras, 11.4ms
Speed: 9.5ms preprocess, 11.4ms inference, 1.4ms postprocess per image at shape (1, 3, 1216, 1216)


100%|██████████| 278/278 [04:16<00:00,  1.08it/s]


✅ Evaluation Summary:
Precision: 0.00%
Recall:    0.00%
F1 Score:  0.00%
→ FP saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_1200/false_positives
→ FN saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_1200/false_negatives
→ MC saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_1200/misclassified


In [ ]:
!pip install ultralytics
import os
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.metrics import precision_score, recall_score, f1_score

# ✅ Paths
yaml_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml"
model_path = "models/yolo11n.pt"
output_dir = "/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_640"

# ✅ Train YOLOv11n model (no resume)
model = YOLO(model_path)
model.train(
    data=yaml_path,
    epochs=300,
    imgsz=640,
    patience=20,
    save=True,
    project=output_dir,
    name="train_yolo11n_reindexed_640"
)

# ✅ Validation and Evaluation
val_images_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images"
val_labels_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/labels"

fp_dir = os.path.join(output_dir, "false_positives")
fn_dir = os.path.join(output_dir, "false_negatives")
mc_dir = os.path.join(output_dir, "misclassified")
os.makedirs(fp_dir, exist_ok=True)
os.makedirs(fn_dir, exist_ok=True)
os.makedirs(mc_dir, exist_ok=True)

class_names = {
    0: "Apoidea", 1: "Arachnida", 2: "Brachycera", 3: "Coleoptera",
    4: "Formicidae", 5: "Nematocera", 6: "Syraphidae"
}

def compute_iou(box1, box2):
    xA, yA = max(box1[0], box2[0]), max(box1[1], box2[1])
    xB, yB = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

print("🔍 Evaluating on validation set...")
y_true, y_pred = [], []

for img_name in tqdm(os.listdir(val_images_path)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace(".jpg", ".txt"))
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    results = model(img_path, conf=0.25, iou=0.7)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()
    pred_classes = results.boxes.cls.cpu().numpy()

    gt_boxes, gt_classes = [], []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w/2) * width)
                y1 = int((y - h/2) * height)
                x2 = int((x + w/2) * width)
                y2 = int((y + h/2) * height)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    matched_pred = set()
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        px1, py1, px2, py2 = map(int, pbox[:4])
        pred_cls = int(pred_classes[i])
        for j, gtbox in enumerate(gt_boxes):
            iou = compute_iou(pbox[:4], gtbox)
            if iou > 0.3:
                matched_pred.add(i)
                matched_gt.add(j)
                if pred_cls != gt_classes[j]:
                    img_copy = image.copy()
                    gx1, gy1, gx2, gy2 = gtbox
                    cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                    cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                    cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                    cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j])}", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                    cv2.imwrite(os.path.join(mc_dir, f"mc_{img_name}"), img_copy)
                break

    for i, pbox in enumerate(pred_boxes):
        if i not in matched_pred:
            y_true.append(0)
            y_pred.append(1)
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            img_copy = image.copy()
            cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
            cv2.putText(img_copy, f"FP: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv2.imwrite(os.path.join(fp_dir, f"fp_{img_name}"), img_copy)

    for j, gtbox in enumerate(gt_boxes):
        if j not in matched_gt:
            y_true.append(1)
            y_pred.append(0)
            gx1, gy1, gx2, gy2 = gtbox
            img_copy = image.copy()
            cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
            cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            cv2.imwrite(os.path.join(fn_dir, f"fn_{img_name}"), img_copy)

# ✅ Summary metrics
precision = precision_score(y_true, y_pred) * 100
recall = recall_score(y_true, y_pred) * 100
f1 = f1_score(y_true, y_pred) * 100

print("\n✅ Evaluation Summary:")
print(f"Precision: {precision:.2f}%")
print(f"Recall:    {recall:.2f}%")
print(f"F1 Score:  {f1:.2f}%")
print(f"→ FP saved to: {fp_dir}")
print(f"→ FN saved to: {fn_dir}")
print(f"→ MC saved to: {mc_dir}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.0/973.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

100%|██████████| 5.35M/5.35M [00:00<00:00, 94.3MB/s]


Ultralytics 8.3.106 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=models/yolo11n.pt, data=/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/config.yaml, epochs=300, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/YOLOv11_Results/reindexed_dataset_run_640, name=train_yolo11n_reindexed_640, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None

100%|██████████| 755k/755k [00:00<00:00, 20.0MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

...

 53%|█████▎    | 146/278 [01:05<00:58,  2.26it/s]


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/1_15077.jpg: 384x640 1 Syraphidae, 9.2ms
Speed: 2.7ms preprocess, 9.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


 53%|█████▎    | 147/278 [01:06<00:59,  2.21it/s]


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images/793_1802938.jpg: 640x640 1 Coleoptera, 9.9ms
Speed: 3.5ms preprocess, 9.9ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)


 53%|█████▎    | 148/278 [01:06<00:54,  2.38it/s]

In [ ]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split
import pandas as pd

# ✅ Paths
background_images_dir = "/content/drive/MyDrive/no-tags-3"  # Folder with 216 background-only .jpg images
existing_train_dir = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/train/images"
existing_val_dir = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_and_split/val/images"

# ✅ New dataset base path
new_dataset_base = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background"
new_train_img_dir = os.path.join(new_dataset_base, "train/images")
new_train_lbl_dir = os.path.join(new_dataset_base, "train/labels")
new_val_img_dir = os.path.join(new_dataset_base, "val/images")
new_val_lbl_dir = os.path.join(new_dataset_base, "val/labels")

os.makedirs(new_train_img_dir, exist_ok=True)
os.makedirs(new_train_lbl_dir, exist_ok=True)
os.makedirs(new_val_img_dir, exist_ok=True)
os.makedirs(new_val_lbl_dir, exist_ok=True)

# ✅ Copy original train/val images and labels to new folders
def copy_dataset(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir):
    for f in os.listdir(src_img_dir):
        if f.endswith(".jpg"):
            shutil.copy(os.path.join(src_img_dir, f), os.path.join(dst_img_dir, f))
            label_file = f.replace(".jpg", ".txt")
            lbl_src = os.path.join(src_lbl_dir, label_file)
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, os.path.join(dst_lbl_dir, label_file))

copy_dataset(existing_train_dir, existing_train_dir.replace("images", "labels"), new_train_img_dir, new_train_lbl_dir)
copy_dataset(existing_val_dir, existing_val_dir.replace("images", "labels"), new_val_img_dir, new_val_lbl_dir)

# ✅ Split background images into 80:20
bg_images = [f for f in os.listdir(background_images_dir) if f.endswith(".jpg")]
train_bg, val_bg = train_test_split(bg_images, test_size=0.2, random_state=42)

# ✅ Copy background images (without labels)
for f in train_bg:
    shutil.copy(os.path.join(background_images_dir, f), os.path.join(new_train_img_dir, f))
for f in val_bg:
    shutil.copy(os.path.join(background_images_dir, f), os.path.join(new_val_img_dir, f))

# ✅ Summary
summary = {
    "Total Background Images": len(bg_images),
    "Background in Train Set": len(train_bg),
    "Background in Val Set": len(val_bg),
    "Total Train Images After Merge": len(os.listdir(new_train_img_dir)),
    "Total Val Images After Merge": len(os.listdir(new_val_img_dir)),
}

df_summary = pd.DataFrame([summary])
df_summary


,Total Background Images,Background in Train Set,Background in Val Set,Total Train Images After Merge,Total Val Images After Merge
0,216,172,44,1282,322


In [ ]:
!pip install ultralytics
import os
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
from sklearn.metrics import precision_score, recall_score, f1_score

# ✅ Paths
yaml_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/config.yaml"
model_path = "models/yolo11n.pt"
output_dir = "/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1000"

# ✅ Train YOLOv11n model (no resume)
model = YOLO(model_path)
model.train(
    data=yaml_path,
    epochs=300,
    imgsz=1000,
    patience=20,
    save=True,
    project=output_dir,
    name="train_yolo11n_reindexed_with_background_1000"
)

# ✅ Validation and Evaluation
val_images_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/val/images"
val_labels_path = "/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/val/labels"

fp_dir = os.path.join(output_dir, "false_positives")
fn_dir = os.path.join(output_dir, "false_negatives")
mc_dir = os.path.join(output_dir, "misclassified")
os.makedirs(fp_dir, exist_ok=True)
os.makedirs(fn_dir, exist_ok=True)
os.makedirs(mc_dir, exist_ok=True)

class_names = {
    0: "Apoidea", 1: "Arachnida", 2: "Brachycera", 3: "Coleoptera",
    4: "Formicidae", 5: "Nematocera", 6: "Syraphidae"
}

def compute_iou(box1, box2):
    xA, yA = max(box1[0], box2[0]), max(box1[1], box2[1])
    xB, yB = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

print("🔍 Evaluating on validation set...")
y_true, y_pred = [], []

for img_name in tqdm(os.listdir(val_images_path)):
    if not img_name.endswith(".jpg"):
        continue

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace(".jpg", ".txt"))
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    results = model(img_path, conf=0.25, iou=0.7)[0]
    pred_boxes = results.boxes.xyxy.cpu().numpy()
    pred_classes = results.boxes.cls.cpu().numpy()

    gt_boxes, gt_classes = [], []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.strip().split())
                x1 = int((x - w/2) * width)
                y1 = int((y - h/2) * height)
                x2 = int((x + w/2) * width)
                y2 = int((y + h/2) * height)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(int(cls))

    matched_pred = set()
    matched_gt = set()

    for i, pbox in enumerate(pred_boxes):
        px1, py1, px2, py2 = map(int, pbox[:4])
        pred_cls = int(pred_classes[i])
        for j, gtbox in enumerate(gt_boxes):
            iou = compute_iou(pbox[:4], gtbox)
            if iou > 0.3:
                matched_pred.add(i)
                matched_gt.add(j)
                if pred_cls != gt_classes[j]:
                    img_copy = image.copy()
                    gx1, gy1, gx2, gy2 = gtbox
                    cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (0, 255, 0), 2)
                    cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
                    cv2.putText(img_copy, f"Pred: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                    cv2.putText(img_copy, f"GT: {class_names.get(gt_classes[j])}", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
                    cv2.imwrite(os.path.join(mc_dir, f"mc_{img_name}"), img_copy)
                break

    for i, pbox in enumerate(pred_boxes):
        if i not in matched_pred:
            y_true.append(0)
            y_pred.append(1)
            px1, py1, px2, py2 = map(int, pbox[:4])
            pred_cls = int(pred_classes[i])
            img_copy = image.copy()
            cv2.rectangle(img_copy, (px1, py1), (px2, py2), (0, 0, 255), 2)
            cv2.putText(img_copy, f"FP: {class_names.get(pred_cls)}", (px1, py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv2.imwrite(os.path.join(fp_dir, f"fp_{img_name}"), img_copy)

    for j, gtbox in enumerate(gt_boxes):
        if j not in matched_gt:
            y_true.append(1)
            y_pred.append(0)
            gx1, gy1, gx2, gy2 = gtbox
            img_copy = image.copy()
            cv2.rectangle(img_copy, (gx1, gy1), (gx2, gy2), (255, 0, 0), 2)
            cv2.putText(img_copy, "FN", (gx1, gy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            cv2.imwrite(os.path.join(fn_dir, f"fn_{img_name}"), img_copy)

# ✅ Summary metrics
precision = precision_score(y_true, y_pred) * 100
recall = recall_score(y_true, y_pred) * 100
f1 = f1_score(y_true, y_pred) * 100

print("\n✅ Evaluation Summary:")
print(f"Precision: {precision:.2f}%")
print(f"Recall:    {recall:.2f}%")
print(f"F1 Score:  {f1:.2f}%")
print(f"→ FP saved to: {fp_dir}")
print(f"→ FN saved to: {fn_dir}")
print(f"→ MC saved to: {mc_dir}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.0/973.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

100%|██████████| 5.35M/5.35M [00:00<00:00, 15.7MB/s]


Ultralytics 8.3.106 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=models/yolo11n.pt, data=/content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/config.yaml, epochs=300, time=None, patience=20, batch=16, imgsz=1000, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1000, name=train_yolo11n_reindexed_with_background_1000, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, r

100%|██████████| 755k/755k [00:00<00:00, 3.24MB/s]


Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

...


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/val/images/2025-03-21_06-53-19-783117_pos3.3_rate4_imx708_0.jpg.jpg: 768x1024 (no detections), 12.7ms
Speed: 6.8ms preprocess, 12.7ms inference, 0.6ms postprocess per image at shape (1, 3, 768, 1024)

image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/val/images/2025-03-14_08-00-28-549463_pos3.4_rate4_imx708_0.jpg.jpg: 1024x1024 (no detections), 14.8ms
Speed: 10.6ms preprocess, 14.8ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 1024)


100%|█████████▉| 321/322 [00:42<00:00,  6.63it/s]


image 1/1 /content/drive/MyDrive/hand-picked-data/2024-04-11-reindexed_with_background/val/images/2025-03-08_10-06-47-260219_pos3.475_rate4_imx708_0.jpg.jpg: 1024x1024 (no detections), 14.1ms
Speed: 6.8ms preprocess, 14.1ms inference, 0.6ms postprocess per image at shape (1, 3, 1024, 1024)


100%|██████████| 322/322 [00:42<00:00,  7.60it/s]


✅ Evaluation Summary:
Precision: 0.00%
Recall:    0.00%
F1 Score:  0.00%
→ FP saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1000/false_positives
→ FN saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1000/false_negatives
→ MC saved to: /content/drive/MyDrive/YOLOv11_Results/reindexed_with_background_1000/misclassified
